# microtom_recon one-zip v0.2.2 — Kaggle
Input 에 (1) 이 zip 의 Dataset, (2) 사진 Dataset 을 추가하고, Settings 에서 **Accelerator = GPU T4 x2**, **Internet = On** 으로 둔 뒤 Run All.
결과는 `/kaggle/working/recon_out/`. 검증되지 않은 결과입니다(자동 마스크, 규모 미검증).
복엽 라벨(7단계)은 `skeleton_v1.json` 과 그룹·축 분류 Dataset(`TPROX_INPUTS_META.json`)이 Input 에 함께 있을 때만 실행됩니다 — README_KO.md 7절.

In [ ]:
import glob, os, shutil, zipfile
CODE = "/kaggle/tmp/onezip"; os.makedirs(CODE, exist_ok=True)
hit = glob.glob("/kaggle/input/**/run_recon.py", recursive=True)             # Kaggle may have unpacked the zip already
if hit:
    shutil.copytree(os.path.dirname(hit[0]), CODE, dirs_exist_ok=True)
else:
    z = [p for p in glob.glob("/kaggle/input/**/*.zip", recursive=True) if "run_recon.py" in zipfile.ZipFile(p).namelist()]
    assert z, "microtom_recon_onezip zip 을 Input 에 추가하세요"
    zipfile.ZipFile(z[0]).extractall(CODE)
print("code:", CODE, sorted(os.listdir(CODE)))

In [ ]:
import glob, os, subprocess, sys
sk = glob.glob("/kaggle/input/**/skeleton_v1.json", recursive=True); ti = glob.glob("/kaggle/input/**/TPROX_INPUTS_META.json", recursive=True)
extra = (["--skeleton", sk[0]] if sk else []) + (["--tprox-inputs", os.path.dirname(ti[0])] if ti else [])   # stage 7 only when both are attached
print("stage 7 inputs:", extra or "none (stages 1-6 only)")
subprocess.run([sys.executable, "/kaggle/tmp/onezip/run_recon.py", "--photos", "/kaggle/input", "--out", "/kaggle/working/recon_out", "--plant-id", "plant", *extra])

In [ ]:
from IPython.display import Image, Markdown, display
import os
O = "/kaggle/working/recon_out"
for f in ("preview.png", "REPORT_KO.md", "masks_sheet.jpg", "REPORT_TPROX_KO.md"):
    if os.path.exists(f"{O}/{f}"):
        display(Markdown(open(f"{O}/{f}", encoding="utf-8").read()) if f.endswith(".md") else Image(f"{O}/{f}"))
print(open(f"{O}/run_log.txt", encoding="utf-8").read()[-2500:])